# Binary toxicity detection (`is_harmful`)

**Target:** `y = 1` if **any** of `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate` is `1`; else `y = 0`.

**Models:** CNN (baseline) → RNN → LSTM → BiLSTM → BERT-family (`distilbert-base-uncased`).  


In [1]:
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Config
DATA_PATH = "/content/dataset.csv"  # Colab: "/content/dataset.csv" etc.

LABEL_COLS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]
BINARY_COL = "is_harmful"

# Vocab settings
MAX_VOCAB = 25000
MIN_FREQ = 2

# Early stopping
EARLY_STOP_VAL_LOSS_RISES = 2

# Model configs
CNN_CONFIG = {
    "batch_size": 64,
    "epochs": 6,
    "lr": 1e-3,
    "grad_clip": 1.0,
    "max_len": 150,           
    "embed_dim": 200,         
    "num_filters": 128,       
    "filter_sizes": [2, 3, 4, 5],  
    "dropout": 0.5,
}

RNN_CONFIG = {
    "batch_size": 64,
    "epochs": 10,
    "lr": 1e-3,               
    "grad_clip": 0.5,        
    "max_len": 100,          
    "embed_dim": 200,
    "hidden_dim": 256,       
    "num_layers": 2,          
    "dropout": 0.5,          
}

LSTM_CONFIG = {
    "batch_size": 64,
    "epochs": 6,
    "lr": 8e-4,
    "grad_clip": 1.0,
    "max_len": 150,
    "embed_dim": 200,
    "hidden_dim": 192,
    "num_layers": 2,         
    "dropout": 0.5,
}

BILSTM_CONFIG = {
    "batch_size": 64,        
    "epochs": 8,             
    "lr": 8e-4,              
    "grad_clip": 1.0,
    "max_len": 150,
    "embed_dim": 200,
    "hidden_dim": 96,         
    "num_layers": 2,          
    "dropout": 0.5,           
}

NN_MODEL_CONFIG = {
    "cnn": CNN_CONFIG,
    "rnn": RNN_CONFIG,
    "lstm": LSTM_CONFIG,
    "bilstm": BILSTM_CONFIG,
}

BERT_CONFIG = {
    "model_name": "distilbert-base-uncased",
    "batch_size": 16,
    "epochs": 2,              
    "lr": 2e-5,
    "max_len": 150,
    "grad_clip": 1.0,
    "warmup_ratio": 0.1,
}

BERT_MODEL = BERT_CONFIG["model_name"]

VAL_SIZE = 0.15
TEST_SIZE = 0.15

## 1. Load data & binary label

In [ ]:
# Load and preprocess data
df = pd.read_csv(DATA_PATH)
df[BINARY_COL] = (df[LABEL_COLS].sum(axis=1) > 0).astype(np.int64)

text_col = "comment_text"
df = df.dropna(subset=[text_col])
df[text_col] = df[text_col].astype(str)

print(df.shape)
print(df[BINARY_COL].value_counts())
df.head(2)

## 2. Preprocess & vocabulary (word-level)

In [ ]:
import html
import unicodedata

# Patterns to identify and replace with placeholders
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", flags=re.IGNORECASE)
USER_RE = re.compile(r"(?<!\w)@\w+", flags=re.IGNORECASE)
IP_RE = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
HTML_TAG_RE = re.compile(r"<[^>]+>")
WIKI_META_RE = re.compile(r"\(\s*talk\s*(?:-\s*contrib)?\s*\)", flags=re.IGNORECASE)
REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}") 
REPEAT_PUNCT_RE = re.compile(r"([!?.,])\1{2,}")

CONTRACTIONS = {
    "won't": "will not",
    "can't": "cannot",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'ve": " have",
    "'m": " am",
}

# Preprocessing functions
def preprocess_text_nn(text: str) -> str:
    """Aggressive but label-safe normalization for word-level models.

    Strategy: normalize unicode, strip HTML, replace high-variance patterns with placeholders,
    keep simple punctuation tokens (!, ?) and collapse repeats.
    """
    text = str(text)
    text = text.strip('"')
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    text = HTML_TAG_RE.sub(" ", text)
    text = WIKI_META_RE.sub(" ", text)
    text = URL_RE.sub(" <url> ", text)
    text = EMAIL_RE.sub(" <email> ", text)
    text = USER_RE.sub(" <user> ", text)
    text = IP_RE.sub(" <ip> ", text)

    text = text.lower()
    for contr, full in CONTRACTIONS.items():
        text = text.replace(contr, full)

    # Keep ! ? as tokens (often signal intensity), normalize repeats
    text = REPEAT_PUNCT_RE.sub(r"\1\1", text)
    text = re.sub(r"([!?])", r" \1 ", text)

    # Replace standalone numbers with a placeholder
    text = re.sub(r"\b\d+(?:[.,]\d+)?\b", " <num> ", text)

    # Keep only letters/digits/apostrophes, our placeholders (< >), and ! ?
    text = re.sub(r"[^a-z0-9\s'<>!?]", " ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)  # e.g. "hellooooo" -> "helloo"

    # Drop stray angle brackets that are not part of our placeholders
    text = re.sub(r"(?<!<)(?:<|>)(?![a-z]+>)", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_text_bert(text: str) -> str:
    """Minimal normalization for BERT to avoid harming tokenization semantics."""
    text = str(text)
    text = text.strip('"')
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str):
    return preprocess_text_nn(text).split()


PAD, UNK = "<pad>", "<unk>"
PAD_IDX, UNK_IDX = 0, 1


def build_vocab(texts, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ):
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize(t))
    specials = [PAD, UNK]
    pairs = [(w, c) for w, c in cnt.most_common() if c >= min_freq]
    pairs = pairs[: max(0, max_vocab - len(specials))]
    itos = specials + [w for w, _ in pairs]
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos


def encode_cnn(text: str, stoi: dict, max_len: int):
    """Encoder for CNN. Length is not used by CNN, but we still return it."""
    ids = [stoi.get(w, UNK_IDX) for w in tokenize(text)]
    if len(ids) >= max_len:
        ids = ids[:max_len]
        length = max_len
    else:
        length = len(ids)
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids, length

# For RNN/LSTM/BiLSTM, we need to guarantee length >= 1 to avoid issues with pack_padded_sequence
def encode_seq(text: str, stoi: dict, max_len: int):
    """Encoder for sequence models (RNN/LSTM/BiLSTM). Guarantees length >= 1."""
    ids = [stoi.get(w, UNK_IDX) for w in tokenize(text)]
    if len(ids) == 0:
        ids = [UNK_IDX]
    if len(ids) >= max_len:
        ids = ids[:max_len]
        length = max_len
    else:
        length = len(ids)
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids, length


train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=df[BINARY_COL],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=temp_df[BINARY_COL],
)

stoi, itos = build_vocab(train_df[text_col].tolist())
vocab_size = len(itos)
print("vocab_size:", vocab_size, "train/val/test:", len(train_df), len(val_df), len(test_df))

## 3. PyTorch `Dataset` for CNN / RNN / LSTM

In [ ]:
# Datasets and loaders
class TabularTextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, text_col: str, label_col: str, stoi: dict, max_len: int, encoder_fn):
        self.texts = frame[text_col].tolist()
        self.labels = frame[label_col].astype(np.float32).values
        self.stoi = stoi
        self.max_len = max_len
        self.encoder_fn = encoder_fn

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = self.encoder_fn(self.texts[idx], self.stoi, self.max_len)
        x = torch.tensor(ids, dtype=torch.long)
        length = min(length, self.max_len)
        y = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return x, torch.tensor(length, dtype=torch.long), y


def collate_batch(batch):
    xs, lens, ys = zip(*batch)
    x_batch = torch.stack(xs, dim=0)
    lens = torch.stack(lens, dim=0)
    y_batch = torch.stack(ys, dim=0)
    return x_batch, lens, y_batch


def collate_batch_seq(batch):
    x_batch, lens, y_batch = collate_batch(batch)
    lens = torch.clamp(lens, min=1)
    return x_batch, lens, y_batch


def make_nn_loaders(model_key: str):
    """Build train/val/test loaders using per-model `max_len` and `batch_size`."""
    cfg = NN_MODEL_CONFIG[model_key.lower()]
    mlen = cfg["max_len"]
    bs = cfg["batch_size"]
    if model_key.lower() == "cnn":
        enc = encode_cnn
        collate = collate_batch
    else:
        enc = encode_seq
        collate = collate_batch_seq
    train_ds = TabularTextDataset(train_df, text_col, BINARY_COL, stoi, mlen, enc)
    val_ds = TabularTextDataset(val_df, text_col, BINARY_COL, stoi, mlen, enc)
    test_ds = TabularTextDataset(test_df, text_col, BINARY_COL, stoi, mlen, enc)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, collate_fn=collate)
    return train_loader, val_loader, test_loader

## 4. Models

In [ ]:
# Models
#CNN
class TextCNN(nn.Module):
    """Kim-style CNN baseline."""

    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, x, lengths=None):
        emb = self.embedding(x).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            h = F.relu(conv(emb))
            pooled.append(h.max(dim=2)[0])
        h = torch.cat(pooled, dim=1)
        h = self.dropout(h)
        return self.fc(h)

#RNN/LSTM/BiLSTM
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, rnn_type="rnn", bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        Rnn = nn.RNN if rnn_type == "rnn" else nn.LSTM
        self.rnn = Rnn(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), 1)
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        out, hidden = self.rnn(packed)
        if self.rnn_type == "rnn":
            h_n = hidden
        else:
            h_n = hidden[0]
        # h_n: (num_layers * num_directions, batch, hidden)
        if self.bidirectional:
            # last layer forward & backward
            fwd = h_n[-2]
            bwd = h_n[-1]
            last = torch.cat([fwd, bwd], dim=1)
        else:
            last = h_n[-1]
        last = self.dropout(last)
        return self.fc(last)

# Model builder
def build_model(name: str, cfg: dict):
    name = name.lower()
    if name == "cnn":
        return TextCNN(
            vocab_size,
            cfg["embed_dim"],
            cfg["num_filters"],
            cfg["filter_sizes"],
            cfg["dropout"],
        )
    if name == "rnn":
        return TextRNN(
            vocab_size,
            cfg["embed_dim"],
            cfg["hidden_dim"],
            cfg["num_layers"],
            cfg["dropout"],
            "rnn",
            bidirectional=False,
        )
    if name == "lstm":
        return TextRNN(
            vocab_size,
            cfg["embed_dim"],
            cfg["hidden_dim"],
            cfg["num_layers"],
            cfg["dropout"],
            "lstm",
            bidirectional=False,
        )
    if name == "bilstm":
        return TextRNN(
            vocab_size,
            cfg["embed_dim"],
            cfg["hidden_dim"],
            cfg["num_layers"],
            cfg["dropout"],
            "lstm",
            bidirectional=True,
        )
    raise ValueError(name)

## 5. Train / evaluate (neural bag-of-conv / RNN)

In [ ]:
# Training and evaluation functions
def run_epoch(model, loader, optimizer=None, grad_clip: float = 1.0):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    all_probs = []
    all_y = []
    crit = nn.BCEWithLogitsLoss()
    for xb, lens, yb in tqdm(loader, leave=False):
        xb, lens, yb = xb.to(device), lens.to(device), yb.to(device)
        if train_mode:
            optimizer.zero_grad()
        logits = model(xb, lens)
        loss = crit(logits, yb)
        if train_mode:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        all_probs.append(probs)
        all_y.append(yb.detach().cpu().numpy().ravel())
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)
    return total_loss / len(loader.dataset), all_probs, all_y


def train_nn(model_name: str):
    model_name = model_name.lower()
    cfg = NN_MODEL_CONFIG[model_name]
    train_loader, val_loader, test_loader = make_nn_loaders(model_name)
    model = build_model(model_name, cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=1
    )
    epochs = cfg["epochs"]
    gclip = cfg.get("grad_clip", 1.0)
    best_auc = -1.0
    best_state = None
    prev_va_loss = None
    val_loss_rise_streak = 0
    for epoch in range(epochs):
        tr_loss, _, _ = run_epoch(model, train_loader, opt, grad_clip=gclip)
        va_loss, va_p, va_y = run_epoch(model, val_loader, None, grad_clip=gclip)
        try:
            auc = roc_auc_score(va_y, va_p)
        except ValueError:
            auc = float("nan")
        current_lr = opt.param_groups[0]["lr"]
        print(
            f"[{model_name}] epoch {epoch+1}/{epochs} train_loss={tr_loss:.4f} "
            f"val_loss={va_loss:.4f} val_auc={auc:.4f} lr={current_lr:.2e}"
        )
        scheduler.step(auc)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if prev_va_loss is not None and va_loss > prev_va_loss:
            val_loss_rise_streak += 1
        else:
            val_loss_rise_streak = 0
        prev_va_loss = va_loss
        if val_loss_rise_streak >= EARLY_STOP_VAL_LOSS_RISES:
            print(
                f"[{model_name}] early stopping at epoch {epoch+1} "
                f"(val loss rose {EARLY_STOP_VAL_LOSS_RISES} epochs in a row)"
            )
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, test_loader


def evaluate_binary(y_true, probs, threshold=0.5, name="model"):
    y_pred = (probs >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n=== {name} — Test set ===")
    print(f"Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")
    print("Confusion matrix [ [TN FP] [FN TP] ]:\n", cm)
    print(classification_report(y_true, y_pred, digits=4))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc}

### 5.1 Train CNN (baseline)

In [ ]:
model_cnn, test_loader_cnn = train_nn("cnn")
_, test_probs_cnn, test_y_cnn = run_epoch(model_cnn, test_loader_cnn, None)
metrics_cnn = evaluate_binary(test_y_cnn, test_probs_cnn, name="CNN (baseline)")

### 5.2 Train RNN

In [ ]:
model_rnn, test_loader_rnn = train_nn("rnn")
_, test_probs_rnn, test_y_rnn = run_epoch(model_rnn, test_loader_rnn, None)
metrics_rnn = evaluate_binary(test_y_rnn, test_probs_rnn, name="RNN")

### 5.3 Train LSTM

In [ ]:
model_lstm, test_loader_lstm = train_nn("lstm")
_, test_probs_lstm, test_y_lstm = run_epoch(model_lstm, test_loader_lstm, None)
metrics_lstm = evaluate_binary(test_y_lstm, test_probs_lstm, name="LSTM")

### 5.4 Train BiLSTM

In [ ]:
model_bilstm, test_loader_bilstm = train_nn("bilstm")
_, test_probs_bilstm, test_y_bilstm = run_epoch(model_bilstm, test_loader_bilstm, None)
metrics_bilstm = evaluate_binary(test_y_bilstm, test_probs_bilstm, name="BiLSTM")

## 6. BERT(`DistilBERT`)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)


class BertBinaryDataset(Dataset):
    """Fast BERT dataset.

    Speedup comes from pre-tokenizing once in __init__ (instead of per-sample __getitem__),
    so the DataLoader only does tensor indexing.

    This does NOT change the semantic inputs (same preprocess + truncation + max_length padding).
    """

    def __init__(self, frame: pd.DataFrame, text_col: str, label_col: str, tokenizer, max_len: int, chunk_size: int = 2048):
        texts = [preprocess_text_bert(t) for t in frame[text_col].astype(str).tolist()]
        labels = frame[label_col].astype(np.int64).values

        # Tokenize in chunks to avoid large peak RAM.
        enc_chunks = []
        for i in range(0, len(texts), chunk_size):
            enc = tokenizer(
                texts[i : i + chunk_size],
                truncation=True,
                padding="max_length",
                max_length=max_len,
                return_tensors="pt",
            )
            enc_chunks.append(enc)

        keys = enc_chunks[0].keys()
        self.encodings = {k: torch.cat([c[k] for c in enc_chunks], dim=0) for k in keys}
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


# DataLoader tuning (Colab GPU): workers + pinned memory
def _bert_num_workers():
    # On Windows/Jupyter local, num_workers>0 can be flaky; keep it safe.
    if torch.cuda.is_available():
        return 2
    return 0


BERT_NUM_WORKERS = _bert_num_workers()
BERT_PIN_MEMORY = torch.cuda.is_available()

bert_train_ds = BertBinaryDataset(train_df, text_col, BINARY_COL, tokenizer, BERT_CONFIG["max_len"])
bert_val_ds = BertBinaryDataset(val_df, text_col, BINARY_COL, tokenizer, BERT_CONFIG["max_len"])
bert_test_ds = BertBinaryDataset(test_df, text_col, BINARY_COL, tokenizer, BERT_CONFIG["max_len"])

bert_train_loader = DataLoader(
    bert_train_ds,
    batch_size=BERT_CONFIG["batch_size"],
    shuffle=True,
    num_workers=BERT_NUM_WORKERS,
    pin_memory=BERT_PIN_MEMORY,
    persistent_workers=(BERT_NUM_WORKERS > 0),
)
bert_val_loader = DataLoader(
    bert_val_ds,
    batch_size=BERT_CONFIG["batch_size"],
    shuffle=False,
    num_workers=BERT_NUM_WORKERS,
    pin_memory=BERT_PIN_MEMORY,
    persistent_workers=(BERT_NUM_WORKERS > 0),
)
bert_test_loader = DataLoader(
    bert_test_ds,
    batch_size=BERT_CONFIG["batch_size"],
    shuffle=False,
    num_workers=BERT_NUM_WORKERS,
    pin_memory=BERT_PIN_MEMORY,
    persistent_workers=(BERT_NUM_WORKERS > 0),
)

In [ ]:
# BERT model and training loop
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)
bert_model.to(device)

bert_opt = torch.optim.AdamW(bert_model.parameters(), lr=BERT_CONFIG["lr"])
total_steps = len(bert_train_loader) * BERT_CONFIG["epochs"]
scheduler = get_linear_schedule_with_warmup(
    bert_opt,
    num_warmup_steps=int(BERT_CONFIG["warmup_ratio"] * total_steps),
    num_training_steps=total_steps,
)

best_auc = -1.0
best_state = None
prev_va_loss = None
val_loss_rise_streak = 0

for epoch in range(BERT_CONFIG["epochs"]):
    bert_model.train()
    running = 0.0
    for batch in tqdm(bert_train_loader, desc=f"BERT train ep{epoch+1}"):
        # non_blocking works with pin_memory=True
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        bert_opt.zero_grad(set_to_none=True)
        out = bert_model(**batch)
        loss = out.loss
        loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), BERT_CONFIG["grad_clip"])
        bert_opt.step()
        scheduler.step()
        running += loss.item() * batch["labels"].size(0)

    bert_model.eval()
    probs_all, y_all = [], []
    val_running = 0.0
    with torch.no_grad():
        for batch in bert_val_loader:
            y_all.append(batch["labels"].numpy())
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            out = bert_model(**batch)
            val_running += out.loss.item() * batch["labels"].size(0)
            prob_pos = F.softmax(out.logits, dim=-1)[:, 1].detach().cpu().numpy()
            probs_all.append(prob_pos)

    probs_all = np.concatenate(probs_all)
    y_all = np.concatenate(y_all)
    va_loss = val_running / len(bert_val_ds)
    try:
        auc = roc_auc_score(y_all, probs_all)
    except ValueError:
        auc = float("nan")
    print(
        f"[BERT] epoch {epoch+1}/{BERT_CONFIG['epochs']} train_loss={running/len(bert_train_ds):.4f} "
        f"val_loss={va_loss:.4f} val_auc={auc:.4f}"
    )

    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.cpu().clone() for k, v in bert_model.state_dict().items()}

    if prev_va_loss is not None and va_loss > prev_va_loss:
        val_loss_rise_streak += 1
    else:
        val_loss_rise_streak = 0
    prev_va_loss = va_loss

    if val_loss_rise_streak >= EARLY_STOP_VAL_LOSS_RISES:
        print(
            f"[BERT] early stopping at epoch {epoch+1} "
            f"(val loss rose {EARLY_STOP_VAL_LOSS_RISES} epochs in a row)"
        )
        break

if best_state is not None:
    bert_model.load_state_dict(best_state)
    bert_model.to(device)

In [ ]:
bert_model.eval()
probs_te, y_te = [], []
with torch.no_grad():
    for batch in tqdm(bert_test_loader, desc="BERT test"):
        y_te.append(batch["labels"].numpy())
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = bert_model(**batch).logits
        prob_pos = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
        probs_te.append(prob_pos)

y_te = np.concatenate(y_te)
probs_te = np.concatenate(probs_te)
metrics_bert = evaluate_binary(y_te.astype(float), probs_te, name=f"BERT ({BERT_MODEL})")

## 7. Compare models

In [ ]:
summary = pd.DataFrame(
    [
        {"model": "CNN (baseline)", **metrics_cnn},
        {"model": "RNN", **metrics_rnn},
        {"model": "LSTM", **metrics_lstm},
        {"model": "BiLSTM", **metrics_bilstm},
        {"model": f"BERT ({BERT_MODEL})", **metrics_bert},
    ]
)
summary

### 7.1 Confusion matrices & ROC

In [ ]:
THRESH_VAL = 0.5


VAL_PLOT_MODELS = [
    "CNN (baseline)",
    "RNN",
    "LSTM",
    "BiLSTM",
    f"BERT ({BERT_MODEL})",
]


def _nn_val_probs(model, key: str):
    _, val_loader, _ = make_nn_loaders(key)
    _, p, y = run_epoch(model, val_loader, None, grad_clip=NN_MODEL_CONFIG[key]["grad_clip"])
    return p, y


def _bert_val_probs():
    bert_model.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for batch in bert_val_loader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = bert_model(**batch).logits
            prob_pos = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            probs_all.append(prob_pos)
            y_all.append(labels)
    return np.concatenate(probs_all), np.concatenate(y_all)


val_bundle = {
    "CNN (baseline)": _nn_val_probs(model_cnn, "cnn"),
    "RNN": _nn_val_probs(model_rnn, "rnn"),
    "LSTM": _nn_val_probs(model_lstm, "lstm"),
    "BiLSTM": _nn_val_probs(model_bilstm, "bilstm"),
    f"BERT ({BERT_MODEL})": _bert_val_probs(),
}

# Sanity check: NN validation labels should align (BERT uses the same split)
_y_ref = val_bundle["CNN (baseline)"][1]
for name, (p, y) in val_bundle.items():
    if not np.array_equal(y, _y_ref):
        print(f"Warning: validation labels differ from CNN baseline: {name}")

plot_names = [n for n in VAL_PLOT_MODELS if n in val_bundle]
if not plot_names:
    raise ValueError("VAL_PLOT_MODELS has no overlap with val_bundle keys; check names.")

n = len(plot_names)
cols = min(3, n)
rows = int(np.ceil(n / cols))
fig_cm, axes_cm = plt.subplots(
    rows, cols, figsize=(4 * cols, 3.4 * rows), constrained_layout=True
)
axes_cm = np.atleast_1d(axes_cm).ravel()
for ax, name in zip(axes_cm[:n], plot_names):
    probs, y_true = val_bundle[name]
    y_pred = (probs >= THRESH_VAL).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(f"{name}\n(val)")
for ax in axes_cm[n:]:
    ax.set_visible(False)
plt.suptitle("Validation confusion matrices (rows=true, cols=predicted)", y=1.02, fontsize=12)
plt.show()


In [ ]:
def _roc_plot_xy(fpr, tpr):
    fpr = np.asarray(fpr, dtype=float)
    tpr = np.asarray(tpr, dtype=float)
    order = np.argsort(fpr, kind="mergesort")
    fpr, tpr = fpr[order], tpr[order]
    ff = np.linspace(0.0, 1.0, 2049)
    tt = np.interp(ff, fpr, tpr, left=0.0, right=float(tpr[-1]) if tpr.size else 0.0)
    return ff, tt


fig_roc, ax_roc = plt.subplots(figsize=(7, 5), dpi=140)
ax_roc.plot([0, 1], [0, 1], "k--", lw=1, label="Chance", antialiased=True)
for name in plot_names:
    probs, y_true = val_bundle[name]
    try:
        fpr, tpr, _ = roc_curve(y_true, probs)
        auc_v = roc_auc_score(y_true, probs)
    except ValueError:
        continue
    fx, ty = _roc_plot_xy(fpr, tpr)
    ax_roc.plot(fx, ty, lw=2.25, label=f"{name} (AUC={auc_v:.3f})", antialiased=True)
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate")
ax_roc.set_title("Validation ROC comparison")
ax_roc.legend(loc="lower right")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)
ax_roc.grid(False)
ax_roc.set_facecolor("white")
plt.tight_layout()
plt.show()